# Search links in docs

pip install nbformat

In [1]:
import os
import nbformat

def search_text_in_files(directory, search_text_base):
    notebooks_with_text = []
    markdown_files_with_text = []

    # Recorre todos los archivos de la carpeta
    for root, dirs, files in os.walk(directory):
        for file in files:
            file_path = os.path.join(root, file)
            if file.endswith(".ipynb"):
                with open(file_path, 'r', encoding='utf-8') as f:
                    nb = nbformat.read(f, as_version=4)
                    
                    # Recorre todas las celdas del notebook
                    for cell in nb.cells:
                        if cell.cell_type == 'markdown':
                            if search_text_base in cell.source:
                                notebooks_with_text.append(file_path)
                                break
            elif file.endswith(".md"):
                with open(file_path, 'r', encoding='utf-8') as f:
                    content = f.read()
                    if search_text_base in content:
                        markdown_files_with_text.append(file_path)

    return notebooks_with_text, markdown_files_with_text

# Especifica la carpeta que contiene los notebooks y el texto a buscar
directory_path = '../docs/'
search_texts = [
    "https://skforecast.org/latest/introduction-forecasting/forecaster-parameters",
    "https://skforecast.org/latest/introduction-forecasting/forecaster-attributes",
    "https://skforecast.org/latest/user_guides/quick-start-skforecast",
    "https://skforecast.org/latest/user_guides/input-data",
    "https://skforecast.org/0.",
    "https://skforecast.org/0.12.1/user_guides/plot-forecaster-residuals",
    "https://skforecast.org/latest/user_guides/plot-forecaster-residuals"
]
search_texts = [
    "py72-classification-forecasting#Probability_calibration"
]

for search_text_base in search_texts:

    print(search_text_base)
    print("=" * len(search_text_base))

    # Llama a la función y obtén los notebooks que contienen el texto buscado
    notebooks, markdown_files = search_text_in_files(directory_path, search_text_base)

    # Imprime la lista de notebooks que contienen el texto
    if notebooks:
        print("The following notebooks contain the specified text:")
        for notebook in notebooks:
            print(notebook)
    else:
        print("No notebooks contain the specified text.")

    if markdown_files:
        print("\nThe following markdown files contain the specified text:")
        for markdown_file in markdown_files:
            print(markdown_file)
    else:
        print("No markdown files contain the specified text.")

    print("=" * len(search_text_base))
    print("\n")


py72-classification-forecasting#Probability_calibration
No notebooks contain the specified text.
No markdown files contain the specified text.




# Test links docs

In [1]:
base_url = "https://ai.skforecast.org/0.1.0/"
# base_url = "https://dev.skforecast.org/0.17.0/"

# Lista de rutas extraídas del archivo mkdocs.yml
paths_general = [
    "",

    "quick-start/quick-start.md",
    "quick-start/first-forecast.md",
    "quick-start/how-to-install.md",

    "user-guides/agentic-forecasting.ipynb",
    "user-guides/agentic-forecasting-step-by-step.ipynb",
    "user-guides/cli-usage.md",

    "api/assistant.md",
    "api/schemas/results.md",
    "api/schemas/plans.md",
    "api/schemas/profiles.md",
    "api/cli.md",

    "releases/releases.md",

    "more/about-skforecast-ai.md",
    "more/consulting.md",
]

paths_user_guides = [
]

# base_url = "https://dev.skforecast.org/0.19.0/"
# paths_general = [
#     "",
#     "introduction-forecasting/introduction-forecasting.md",
#     "user_guides/table-of-contents.md",
#     "faq/table-of-contents.md",
#     "quick-start/forecaster-parameters.md",
#     "releases/releases.md",
# ]


# Función para cambiar la extensión a .html
def change_extension_to_html(path):
    if path.endswith(".md") or path.endswith(".ipynb"):
        return path.rsplit(".", 1)[0] + ".html"
    return path


# Generar enlaces completos con extensión .html
links_general = [
    base_url + change_extension_to_html(path) for path in paths_general
]
links_user_guides = [
    base_url + change_extension_to_html(path) for path in paths_user_guides
]

In [2]:
from colorama import Fore, Style

print("Esto es un texto normal")
print(Fore.RED + "Este texto es rojo" + Style.RESET_ALL)
print(Fore.GREEN + "Este texto es verde" + Style.RESET_ALL)
print(Fore.BLUE + "Este texto es azul" + Style.RESET_ALL)
print(Style.BRIGHT + Fore.RED + "Este texto es rojo en negrita" + Style.RESET_ALL)
print("Esto es un texto normal")

Esto es un texto normal
Este texto es rojo
Este texto es verde
Este texto es azul
Este texto es rojo en negrita
Esto es un texto normal


In [3]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from colorama import Fore, Style


def check_links_ignoring_nav_links(url):
    # 1. Usar Session para mantener conexiones abiertas (reutiliza el handshake SSL)
    session = requests.Session()
    # User-Agent para parecer un navegador real
    session.headers.update({'User-Agent': 'Mozilla/5.0 (Link Checker)'})

    try:
        response = session.get(url, timeout=10)
        response.raise_for_status()
        print(f"[OK] Página accesible: {url}")
    except requests.exceptions.RequestException as e:
        print(Style.BRIGHT + Fore.RED + f"[ERROR] No se pudo acceder a {url}: {e}" + Style.RESET_ALL)
        return [(url, url, str(e))]  # (source_url, broken_link, error)

    # Parsear el contenido HTML
    soup = BeautifulSoup(response.content, "html.parser")
    links = soup.find_all("a", href=True)

    classes_no_to_visit = [
        'md-header__button', 'md-logo', "md-nav__link", "md-tabs__link",
        'md-source', 'md-social__link', 'autorefs-external'
    ]

    # Primera pasada: Solo recolectar URLs válidas
    links_to_check = []
    for link in links:
        class_link = link.get("class", [])
        
        # Filtros de clases
        if any(c in class_link for c in classes_no_to_visit):
            continue

        href = link['href']
        
        # Ignorar anclas internas, mailto, etc.
        if href.startswith(("#", "mailto:", "tel:", "javascript:")):
            continue

        full_url = urljoin(url, href)
        links_to_check.append(full_url)

    print(f"Comprobando {len(links_to_check)} enlaces...")
    
    broken_links = []

    # Segunda pasada: Comprobar enlaces uno a uno
    for link_url in links_to_check:
        try:
            # 2. Intentamos HEAD primero (mucho más rápido, no descarga el cuerpo)
            # allow_redirects=True es importante para seguir redirecciones
            r = session.head(link_url, timeout=5, allow_redirects=True)
            
            # Algunos servidores devuelven 405 o 404 a HEAD pero funcionan con GET
            # Si el status es de error (>=400), probamos con GET por si acaso
            if r.status_code >= 400:
                r = session.get(link_url, timeout=5, stream=True) 
                # stream=True descarga solo headers inicialmente, ahorrando ancho de banda

            if r.status_code >= 400:
                raise requests.exceptions.RequestException(f"Status {r.status_code}")

            print(f"  [OK] {link_url}")

        except requests.exceptions.RequestException as e:
            print(
                Style.BRIGHT + Fore.RED + 
                f"  [ERROR] Roto: {link_url} ({e})" 
                + Style.RESET_ALL
            )
            broken_links.append((url, link_url, str(e)))  # (source_url, broken_link, error)

    return broken_links

# Ejemplo de uso
# url = "https://skforecast.org/0.14.0/user_guides/table-of-contents.html"
# check_links_ignoring_nav_links(url)

In [4]:
# Check all links in user guides list
# ==============================================================================
from collections import defaultdict

broken_links_user_guides = []
for link in links_user_guides:
    broken_links = check_links_ignoring_nav_links(link)
    broken_links_user_guides.extend(broken_links)

print("")

# Resumen de enlaces rotos
# ==============================================================================
print(Style.BRIGHT + Fore.RED + "\n" + "=" * 80)
print("RESUMEN DE ENLACES ROTOS")
print("=" * 80)

grouped = defaultdict(list)
for source_url, broken_link, error in broken_links_user_guides:
    grouped[source_url].append((broken_link, error))

for source_url, links in grouped.items():
    print(f"\n[Origen] {source_url}")
    for link, error in links:
        print(f"   [{error}] {link}")

print("\n" + "=" * 80 + Style.RESET_ALL)



RESUMEN DE ENLACES ROTOS



In [5]:
# Check all links in general list
# ==============================================================================
from collections import defaultdict

broken_links_general = []
for link in links_general:
    broken_links = check_links_ignoring_nav_links(link)
    broken_links_general.extend(broken_links)

print("")

# Resumen de enlaces rotos
# ==============================================================================
print(Style.BRIGHT + Fore.RED + "\n" + "=" * 80)
print("RESUMEN DE ENLACES ROTOS")
print("=" * 80)

grouped = defaultdict(list)
for source_url, broken_link, error in broken_links_general:
    grouped[source_url].append((broken_link, error))

for source_url, links in grouped.items():
    print(f"\n[Origen] {source_url}")
    for link, error in links:
        print(f"   [{error}] {link}")

print("\n" + "=" * 80 + Style.RESET_ALL)

[OK] Página accesible: https://ai.skforecast.org/0.1.0/
Comprobando 25 enlaces...
  [OK] https://ai.skforecast.org/
  [OK] https://pypi.org/project/skforecast-ai/
  [OK] https://github.com/skforecast/skforecast-ai/actions/workflows/unit-tests.yml
  [OK] https://www.repostatus.org/#active
  [ERROR] Roto: https://pepy.tech/project/skforecast-ai (Status 404)
  [OK] https://pypistats.org/packages/skforecast-ai
  [OK] https://github.com/skforecast/skforecast-ai/blob/main/LICENSE
  [OK] https://www.paypal.com/donate/?hosted_button_id=D2JZSWRLTZDL6
  [OK] https://www.buymeacoffee.com/skforecast
  [OK] https://opencollective.com/skforecast
  [OK] https://www.linkedin.com/company/skforecast/
  [OK] https://discord.gg/3V52qpNkuj
  [OK] https://cienciadedatos.net/en/forecasting-python
  [OK] https://studio.skforecast.org/
  [OK] https://skforecast.org
  [OK] https://ai.skforecast.org/0.1.0/quick-start/first-forecast
  [OK] https://ai.skforecast.org/0.1.0/user-guides/cli-usage
  [OK] https://ai.sk